# 1. Snowflake Setup and Export to Ossie (S3 Backbone)

This notebook shows the Snowflake side of the demo: the tables and semantic view created in the previous notebook, and then exports the semantic view to an Apache Ossie file on the same S3 bucket that the Iceberg tables themselves live on. Databricks reads from the same bucket automatically.

Order of work:
1. Show the tables and semantic view in the schema.
2. Export the semantic view to Ossie YAML on the S3 stage.
3. (Optional) Create a suspended sync task for live demo.

## Step 1 - Set your database and schema

In [ ]:
DATABASE = "DEMOS"
SCHEMA   = "DEMO_SEMANTIC_INTEROP"   # semantic view, stage and tasks
DATA_SCHEMA = "EXT_SEMANTIC_INTEROP"   # Iceberg tables, shared with the other flows
SEMANTIC_VIEW_NAME = 'SALES_SV'
STAGE_NAME = 'DEMO_OSSIE_STAGE'
OUTPUT_FILE = 'ossie_from_snowflake.yaml'

print(f"Working in {DATABASE}.{SCHEMA}")

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;
USE SCHEMA {{DATABASE}}.{{SCHEMA}};

## Step 2 - Examine the Data

The tables are Snowflake-managed Iceberg tables stored on S3. Both Snowflake and Databricks read from the same physical Parquet files.

In [ ]:
%%sql -r dataframe_9
SHOW ICEBERG TABLES IN {{DATABASE}}.{{DATA_SCHEMA}};

In [ ]:
%%sql -r dataframe_17
SELECT SYSTEM$GET_ICEBERG_TABLE_INFORMATION('{{DATABASE}}.{{DATA_SCHEMA}}.CUSTOMERS');

In [ ]:
%%sql -r dataframe_10
SELECT 
    *
FROM {{DATABASE}}.{{DATA_SCHEMA}}.CUSTOMERS

In [ ]:
%%sql -r dataframe_11
SELECT 
    *
FROM {{DATABASE}}.{{DATA_SCHEMA}}.ORDERS;

In [ ]:
%%sql -r dataframe_2
-- Expected: EAST 750/5/12, WEST 700/5/11
SELECT 
    c.region, 
    SUM(o.order_amount) AS total_amount, 
    COUNT(o.order_id) AS order_count,
    SUM(o.order_qty) AS total_qty
FROM {{DATABASE}}.{{DATA_SCHEMA}}.ORDERS o
JOIN {{DATABASE}}.{{DATA_SCHEMA}}.CUSTOMERS c USING (customer_id)
GROUP BY c.region ORDER BY c.region;

## Step 3 - The semantic view

Setup already created `SALES_SV`, so this notebook only inspects it. That matters if you
ever step backwards during the demo: re-running a `CREATE OR REPLACE` here would silently
throw away whatever Databricks had just synced back, and the model you are about to export
would be the original rather than the round-tripped one.

Note the synonyms on the dimensions and metrics. They are the business vocabulary that
Cortex Analyst resolves against, and they travel with the model to Databricks.


In [ ]:
%%sql -r dataframe_3
-- Inspect, do not recreate. Setup owns creation (Snowflake/01_setup).
-- If SALES_SV is missing, run that notebook rather than creating it here.
SHOW SEMANTIC VIEWS LIKE '{{SEMANTIC_VIEW_NAME}}' IN SCHEMA {{DATABASE}}.{{SCHEMA}};


<details>
<summary>The DDL that setup ran, for reference</summary>

```sql
CREATE OR REPLACE SEMANTIC VIEW DEMOS.DEMO_SEMANTIC_INTEROP.SALES_SV
  TABLES (
    orders AS DEMOS.EXT_SEMANTIC_INTEROP.ORDERS PRIMARY KEY (order_id),
    customers AS DEMOS.EXT_SEMANTIC_INTEROP.CUSTOMERS PRIMARY KEY (customer_id)
  )
  RELATIONSHIPS (
    orders_to_customers AS orders (customer_id) REFERENCES customers (customer_id)
  )
  FACTS (
    orders.order_amount AS order_amount,
    orders.order_qty AS order_qty
  )
  DIMENSIONS (
    customers.region AS region WITH SYNONYMS ('area', 'territory'),
    customers.customer_name AS customer_name WITH SYNONYMS ('client', 'account')
  )
  METRICS (
    orders.total_order_amount AS SUM(orders.order_amount)
      WITH SYNONYMS ('revenue', 'sales'),
    orders.order_count AS COUNT(orders.order_id)
      WITH SYNONYMS ('orders', 'transactions')
  );
```

The tables are in `EXT_SEMANTIC_INTEROP` while the view is in `DEMO_SEMANTIC_INTEROP`.
The data is shared by all three demo flows; only the semantic model is per-flow.

</details>


In [ ]:
%%sql -r dataframe_4
SELECT * FROM SEMANTIC_VIEW(
  {{DATABASE}}.{{SCHEMA}}.{{SEMANTIC_VIEW_NAME}}
  DIMENSIONS customers.region
  METRICS orders.total_order_amount, orders.order_count
) ORDER BY region;

In [ ]:
%%sql -r dataframe_15
-- DESCRIBE SEMANTIC VIEW {{DATABASE}}.{{SCHEMA}}.{{SEMANTIC_VIEW_NAME}}
SHOW SEMANTIC METRICS IN {{DATABASE}}.{{SCHEMA}}.{{SEMANTIC_VIEW_NAME}};

## Step 4a - Manual Export Of Ossie File

Step 4b sets up automatic exporting, skip ahead if desired.

The COPY writes the Ossie YAML to `s3://<your-bucket>/ossie/demo/ossie_from_snowflake.yaml`.
Databricks reads this file directly from the same bucket.

In [ ]:
%%sql -r dataframe_5
COPY INTO @{{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}}/ossie_from_snowflake.yaml
FROM (
    SELECT SYSTEM$READ_OSSIE_YAML_FROM_SEMANTIC_VIEW('{{DATABASE}}.{{SCHEMA}}.{{SEMANTIC_VIEW_NAME}}')
)
FILE_FORMAT = (TYPE = CSV FIELD_DELIMITER = NONE RECORD_DELIMITER = NONE
               ESCAPE_UNENCLOSED_FIELD = NONE COMPRESSION = NONE)
SINGLE = TRUE OVERWRITE = TRUE;

In [ ]:
%%sql -r dataframe_6
LIST @{{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}};

## Done

The Ossie file is at `s3://<your-bucket>/ossie/demo/ossie_from_snowflake.yaml`.
Open notebook 2 in Databricks -- it reads directly from S3, no file transfer needed.

## Step 4b - Automatic Syncing via Task

Create a stored procedure to handle the Ossie export, and a task to pick up any changes to the Semantic View and trigger an export. It runs every minute, keeping semantics in sync across platforms. 

1. Create a Stored Procedure which exports a Semantic View to Ossie.
2. Create a task that monitors the Semantic View for changes.
3. If changes are detected, the stored procedure exports the updated semantic view.

**Created suspended** -- enable during
demo, disable immediately after.

```sql
ALTER TASK DEMOS.DEMO_SEMANTIC_INTEROP.OSSIE_SYNC_TASK RESUME;
ALTER TASK DEMOS.DEMO_SEMANTIC_INTEROP.OSSIE_SYNC_TASK SUSPEND;
```

## 1. Create a Stored Procedure

In [ ]:
%%sql -r export_proc_result
CREATE OR REPLACE PROCEDURE EXPORT_OSSIE_TO_STAGE(
    P_DATABASE STRING,
    P_SCHEMA STRING,
    P_SEMANTIC_VIEW_NAME STRING,
    P_STAGE_NAME STRING,
    P_OUTPUT_FILE_NAME STRING
)
RETURNS STRING
LANGUAGE SQL
EXECUTE AS OWNER
AS
BEGIN
    LET stage_path STRING := P_DATABASE || '.' || P_SCHEMA || '.'  || P_STAGE_NAME || '/' || P_OUTPUT_FILE_NAME;
    LET fqn STRING := P_DATABASE || '.' || P_SCHEMA || '.' || P_SEMANTIC_VIEW_NAME;

    EXECUTE IMMEDIATE '
        COPY INTO @' || :stage_path || '
        FROM (
            SELECT SYSTEM$READ_OSSIE_YAML_FROM_SEMANTIC_VIEW(''' || :fqn || ''')
        )
        FILE_FORMAT = (TYPE = CSV FIELD_DELIMITER = NONE RECORD_DELIMITER = NONE
                       ESCAPE_UNENCLOSED_FIELD = NONE COMPRESSION = NONE)
        SINGLE = TRUE OVERWRITE = TRUE';

    RETURN 'Exported OSSIE YAML to @' || :stage_path;
END;

In [ ]:
%%sql -r dataframe_19
CALL EXPORT_OSSIE_TO_STAGE(
    '{{DATABASE}}',
    '{{SCHEMA}}',
    '{{SEMANTIC_VIEW_NAME}}',
    '{{STAGE_NAME}}',
    '{{OUTPUT_FILE}}'
);

In [ ]:
%%sql -r dataframe_20
SELECT * FROM 
    DIRECTORY(@{{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}})
WHERE relative_path ILIKE '{{OUTPUT_FILE}}'; 

## Monitoring Task

This is turned off for demo purposes, but would run every minute in production.

In [ ]:
%%sql -r dataframe_23
-- Updated task
CREATE OR REPLACE TASK DEMOS.DEMO_SEMANTIC_INTEROP.MONITOR_SV_CHANGES
  WAREHOUSE = SI_DEMO_WH
  SCHEDULE = '1 MINUTE'
AS
BEGIN
  LET sv_altered TIMESTAMP_LTZ;
  LET file_modified TIMESTAMP_LTZ;

  -- Refresh to populate metadata (needed for external stages)
  ALTER STAGE {{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}} REFRESH;

  SHOW SEMANTIC VIEWS LIKE '{{SEMANTIC_VIEW_NAME}}' IN SCHEMA {{DATABASE}}.{{SCHEMA}};
  SELECT "last_altered" INTO :sv_altered FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

  SELECT LAST_MODIFIED INTO :file_modified
    FROM DIRECTORY(@{{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}})
    WHERE RELATIVE_PATH = '{{OUTPUT_FILE}}';

  IF (:file_modified IS NULL OR :sv_altered > :file_modified) THEN
    CALL EXPORT_OSSIE_TO_STAGE('{{DATABASE}}', '{{SCHEMA}}', '{{SEMANTIC_VIEW_NAME}}', '{{STAGE_NAME}}', '{{OUTPUT_FILE}}');
    ALTER STAGE DEMOS.DEMO_SEMANTIC_INTEROP.DEMO_OSSIE_STAGE REFRESH;
  END IF;
END;    

In [ ]:
%%sql -r dataframe_12
ALTER TASK DEMOS.DEMO_SEMANTIC_INTEROP.MONITOR_SV_CHANGES RESUME;

In [ ]:
%%sql -r dataframe_27
ALTER TASK DEMOS.DEMO_SEMANTIC_INTEROP.MONITOR_SV_CHANGES SUSPEND;

In [ ]:
%%sql -r dataframe_21
SHOW TASKS LIKE 'MONITOR_SV_CHANGES' IN SCHEMA DEMOS.DEMO_SEMANTIC_INTEROP;

In [ ]:
%%sql -r dataframe_7
SELECT * FROM 
    DIRECTORY(@{{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}})
WHERE relative_path ILIKE '{{OUTPUT_FILE}}'; 

In [ ]:
%%sql -r dataframe_18
LIST @{{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}}; -- 18:36